In [117]:
import pandas as pd
import numpy as np

In [3]:
# =============================================================
# HC-WCI DATA EXTRACTION CODE — RECTIFIED VERSION
# Wright State University | Amir Hasan Khan
#
# KEY FIXES FROM SANITY CHECK REPORT:
#   FIX 1: Avg_Rent_Paid now = Sum_Rent / Total_Renters (NOT Total_People)
#           This prevents owners (whose RENT=0) from deflating the rent average
#   FIX 2: State_Median_Rent now computed from renter-only population
#           so it reflects actual market rent levels, not a per-capita burden
#   FIX 3: This eliminates the structural data leakage between Rp and
#           Mortgage_Rate_Pct that existed in the original code
#   FIX 4: Minimum cohort filter (Total_People >= 100) added before saving
#           to suppress degenerate 0% / 100% mortgage rate spikes
#   FIX 5: Mortgage_Rate_Pct renamed to Mortgaged_Ownership_Rate to match
#           what the variable actually measures (% of ENTIRE cohort who are
#           mortgaged homeowners, NOT % of owners who carry a mortgage)
#   NOTE:   Years_in_US >= 0 filter added to remove pre-arrival artefacts
# =============================================================

import pandas as pd
import numpy as np

# ── 1. CONFIGURATION ─────────────────────────────────────────
input_file  = 'raw_data'
output_file = 'HCWCI_Master_Dataset_v2.csv'

country_map = {
    20000: 'Mexico',            26010: 'Dominican Republic',
    50000: 'China',             51500: 'Philippines',
    51800: 'Vietnam',           52100: 'India',
    52110: 'Bangladesh',        54200: 'Turkey',
    60031: 'Nigeria',           60045: 'Kenya'
}

state_map = {
    1:'Alabama', 2:'Alaska', 4:'Arizona', 5:'Arkansas', 6:'California',
    8:'Colorado', 9:'Connecticut', 10:'Delaware', 11:'DC', 12:'Florida',
    13:'Georgia', 15:'Hawaii', 16:'Idaho', 17:'Illinois', 18:'Indiana',
    19:'Iowa', 20:'Kansas', 21:'Kentucky', 22:'Louisiana', 23:'Maine',
    24:'Maryland', 25:'Massachusetts', 26:'Michigan', 27:'Minnesota',
    28:'Mississippi', 29:'Missouri', 30:'Montana', 31:'Nebraska',
    32:'Nevada', 33:'New Hampshire', 34:'New Jersey', 35:'New Mexico',
    36:'New York', 37:'North Carolina', 38:'North Dakota', 39:'Ohio',
    40:'Oklahoma', 41:'Oregon', 42:'Pennsylvania', 44:'Rhode Island',
    45:'South Carolina', 46:'South Dakota', 47:'Tennessee', 48:'Texas',
    49:'Utah', 50:'Vermont', 51:'Virginia', 53:'Washington',
    54:'West Virginia', 55:'Wisconsin', 56:'Wyoming'
}

def get_educ_group(e):
    if e <= 2:   return "1_Low (No/Primary)"
    elif e <= 5: return "2_Some High School"
    elif e == 6: return "3_HS Graduate"
    elif e <= 8: return "4_Some College"
    else:        return "5_College+ (Bachelor or Higher)"


# ── 2. CHUNKED EXTRACTION ────────────────────────────────────
cols = ['YEAR','PERWT','STATEFIP','BPLD','YRIMMIG',
        'OWNERSHP','MORTGAGE','RENT','EDUC','INCTOT']

print(f"Reading {input_file} in chunks...")
processed_list = []
chunk_count = 0

for chunk in pd.read_csv(input_file, usecols=cols, chunksize=100_000):

    # -- income filter: remove top-coded non-response values only --
    # NOTE: zero-income records are KEPT (valid negative/zero earners exist)
    chunk = chunk[chunk['INCTOT'] < 9_999_998].copy()

    # -- country and state filters --
    chunk['Country'] = chunk['BPLD'].map(country_map)
    chunk = chunk.dropna(subset=['Country']).copy()
    chunk = chunk[chunk['YRIMMIG'] > 0].copy()

    if chunk.empty:
        continue

    chunk['State_Name']      = chunk['STATEFIP'].map(state_map)
    chunk['Education_Group'] = chunk['EDUC'].apply(get_educ_group)
    chunk['Years_in_US']     = chunk['YEAR'] - chunk['YRIMMIG']

    # -- residency window: 0–30 years (FIX: also remove negative values) --
    chunk = chunk[(chunk['Years_in_US'] >= 0) & (chunk['Years_in_US'] <= 30)].copy()

    # ── weighted flags ──
    chunk['Has_Mortgage'] = np.where(
        (chunk['OWNERSHP'] == 1) & (chunk['MORTGAGE'] > 1), 1, 0)

    # FIX 1: identify renters to correctly compute rent averages
    # A renter is someone who pays rent (OWNERSHP == 2 OR RENT > 0).
    # We use RENT > 0 as the flag because it is the cleanest observable signal.
    chunk['Is_Renter'] = np.where(chunk['RENT'] > 0, 1, 0)

    chunk['W_Mortgages'] = chunk['Has_Mortgage'] * chunk['PERWT']
    chunk['W_Income']    = chunk['INCTOT']        * chunk['PERWT']
    chunk['W_Rent']      = chunk['RENT']           * chunk['PERWT']
    # FIX 1 continued: weighted renter count
    chunk['W_Renters']   = chunk['Is_Renter']      * chunk['PERWT']

    summary = chunk.groupby(
        ['Country', 'State_Name', 'Years_in_US', 'Education_Group']
    ).agg(
        Total_People    = ('PERWT',        'sum'),
        Sum_Mortgages   = ('W_Mortgages',  'sum'),
        Sum_Income      = ('W_Income',     'sum'),
        Sum_Rent        = ('W_Rent',       'sum'),
        Total_Renters   = ('W_Renters',    'sum'),  # FIX 1: renter count
    ).reset_index()

    processed_list.append(summary)

    chunk_count += 1
    if chunk_count % 10 == 0:
        print(f"  Processed {chunk_count * 100_000:,} rows...")


# ── 3. AGGREGATION ───────────────────────────────────────────
print("Consolidating final dataset...")
final_df = (
    pd.concat(processed_list)
    .groupby(['Country', 'State_Name', 'Years_in_US', 'Education_Group'])
    .sum()
    .reset_index()
)


# ── 4. STATE RENT BENCHMARK — RENTER-ONLY (FIX 2) ────────────
# Original bug: Sum_Rent / Total_People mixed owners (RENT=0) into denominator,
# producing values 40–70 percent below actual state median rents.
# Fix: divide by Total_Renters so the benchmark reflects what renters pay.

state_market = (
    final_df[final_df['Total_Renters'] > 0]
    .groupby('State_Name')
    .apply(lambda x: x['Sum_Rent'].sum() / x['Total_Renters'].sum())
    .reset_index()
)
state_market.columns = ['State_Name', 'State_Rent_Benchmark']  # renamed from State_Median_Rent
final_df = final_df.merge(state_market, on='State_Name', how='left')


# ── 5. DERIVED METRICS ───────────────────────────────────────

# FIX 5: renamed to reflect what is actually measured
final_df['Mortgaged_Ownership_Rate'] = (
    final_df['Sum_Mortgages'] / final_df['Total_People']) * 100

final_df['Avg_Income'] = final_df['Sum_Income'] / final_df['Total_People']

# FIX 1 applied: Avg_Rent_Paid now uses Total_Renters as denominator
final_df['Avg_Rent_Paid_Renters'] = np.where(
    final_df['Total_Renters'] > 0,
    final_df['Sum_Rent'] / final_df['Total_Renters'],
    np.nan
)

# FIX 2 + FIX 3: Rp now reflects renter cash-flow capacity,
# normalized against the renter-based state benchmark.
# This breaks the mechanical link between Rp and Mortgage_Rate_Pct.
final_df['Rent_Performance_Ratio'] = (
    final_df['Avg_Rent_Paid_Renters'] / final_df['State_Rent_Benchmark']
)


# ── 6. QUALITY FILTERS ───────────────────────────────────────
rows_before = len(final_df)

# FIX 4: drop cohorts with fewer than 100 weighted people to remove
# degenerate 0% and 100% mortgage rate artefacts
final_df = final_df[final_df['Total_People'] >= 100].copy()

# Drop cohorts where Rp could not be computed (all owners, no renters)
final_df = final_df.dropna(subset=['Rent_Performance_Ratio']).copy()

rows_after = len(final_df)
print(f"Quality filter removed {rows_before - rows_after:,} cohorts "
      f"(< 100 people or no renter data).")
print(f"Final cohort count: {rows_after:,}")


# ── 7. WINSORIZE Rp OUTLIERS ─────────────────────────────────
rp_cap = final_df['Rent_Performance_Ratio'].quantile(0.95)
rp_outliers = (final_df["Rent_Performance_Ratio"].clip(upper=rp_cap) != final_df["Rent_Performance_Ratio"]).sum()
print(f"Rp winsorized at 95th pct: {rp_cap:.3f} ({rp_outliers:,} outliers capped)")



# ── 8. SAVE ──────────────────────────────────────────────────
final_df.to_csv(output_file, index=False)
print(f"\nDONE. {output_file} saved.")
print(f"Total weighted population: {final_df['Total_People'].sum():,.0f}")
print(f"Survey years covered: 2014–2024 (11 years)")
print(f"States: 51 (50 states + DC)")

Reading raw_data in chunks...
  Processed 1,000,000 rows...
  Processed 2,000,000 rows...
  Processed 3,000,000 rows...
  Processed 4,000,000 rows...
  Processed 5,000,000 rows...
  Processed 6,000,000 rows...
  Processed 7,000,000 rows...
  Processed 8,000,000 rows...
  Processed 9,000,000 rows...
  Processed 10,000,000 rows...
  Processed 11,000,000 rows...
  Processed 12,000,000 rows...
  Processed 13,000,000 rows...
  Processed 14,000,000 rows...
  Processed 15,000,000 rows...
  Processed 16,000,000 rows...
  Processed 17,000,000 rows...
  Processed 18,000,000 rows...
  Processed 19,000,000 rows...
  Processed 20,000,000 rows...
  Processed 21,000,000 rows...
  Processed 22,000,000 rows...
  Processed 23,000,000 rows...
  Processed 24,000,000 rows...
  Processed 25,000,000 rows...
  Processed 26,000,000 rows...
  Processed 27,000,000 rows...
  Processed 28,000,000 rows...
  Processed 29,000,000 rows...
  Processed 30,000,000 rows...
  Processed 31,000,000 rows...
  Processed 32,000

C:\Users\Amirh\AppData\Local\Temp\ipykernel_27164\3558492549.py:137: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x['Sum_Rent'].sum() / x['Total_Renters'].sum())



DONE. HCWCI_Master_Dataset_v2.csv saved.
Total weighted population: 166,261,329
Survey years covered: 2014–2024 (11 years)
States: 51 (50 states + DC)
